# 05 — Metrics and Figure Data

**Purpose.** Compute every quantitative result reported in the paper, sourced from the three result CSVs produced by notebooks `02`, `03`, and `04`. This notebook is the canonical bridge between the raw model outputs (per-sample predictions) and the aggregate numbers shown in the paper's tables, figures, and discussion.

**Inputs.**
- `<BASE_DIR>/results/rq1_ablation_results_hinted.csv` — produced by `02_run_ablation_hinted.ipynb`.
- `<BASE_DIR>/results/rq2_ablation_results_clean.csv` — produced by `03_run_ablation_clean.ipynb`.
- `<BASE_DIR>/results/rq3_cross_generation_gpt35.csv` — produced by `04_run_cross_generation.ipynb`.

**Outputs.**
- `<BASE_DIR>/results/case_study_discordant_cmdinj.csv` — the discordant CmdInj cases used in the qualitative analysis (Section 5.1 of the paper).
- All other outputs are printed inline; the paper's tables and figures are produced from these printed numbers (figure visuals are authored separately in `figures/`).

**Cross-references to the paper.**
- Section 2 of this notebook → Table 1
- Section 3 → §4.1 main McNemar test (p = 0.0003 for Variant E vs Baseline)
- Section 4 → Figure 2 (per-CWE breakdown)
- Section 5 → Table 2 (paired HINTED–CLEAN McNemar tests)
- Section 6 → §4.3 cross-generation finding (98 of 100 Safe predictions on GPT-3.5-turbo)
- Section 7 → §5.1 case study sample (30 discordant CmdInj cases)
- Section 8 → numeric data underlying Figures 1, 2, 3, 4


## 1. Setup — load all three result CSVs

Read the three result CSVs and run sanity checks: the RQ1 and RQ2 tables must each contain exactly 234 rows, and the RQ3 table must contain exactly 100 rows. Any deviation indicates an incomplete upstream notebook run.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)
from statsmodels.stats.contingency_tables import mcnemar

# ---- USER-EDITABLE ----
BASE_DIR_OVERRIDE = None
# -----------------------

def find_repo_root() -> Path:
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT      = find_repo_root()
BASE_DIR       = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')
RESULTS_DIR    = BASE_DIR / 'results'

RQ1_CSV        = RESULTS_DIR / 'rq1_ablation_results_hinted.csv'
RQ2_CSV        = RESULTS_DIR / 'rq2_ablation_results_clean.csv'
RQ3_CSV        = RESULTS_DIR / 'rq3_cross_generation_gpt35.csv'
CASE_STUDY_OUT = RESULTS_DIR / 'case_study_discordant_cmdinj.csv'

# Load.
df_rq1 = pd.read_csv(RQ1_CSV)
df_rq2 = pd.read_csv(RQ2_CSV)
df_rq3 = pd.read_csv(RQ3_CSV)

# Treat 'Error' / NaN as 'Safe' (no false alarm) — this matches the convention used during the original experiments.
VARIANT_COLS_RQ1 = ['Variant_A_Baseline', 'Variant_B_Persona', 'Variant_C_Patterns', 'Variant_D_CoT', 'Variant_E_Full']
VARIANT_COLS_RQ2 = ['Variant_C_Patterns_CLEAN', 'Variant_E_Full_CLEAN']
for col in VARIANT_COLS_RQ1:
    df_rq1[col] = df_rq1[col].fillna('Safe').replace({'Error': 'Safe', 'API_Error': 'Safe'})
for col in VARIANT_COLS_RQ2:
    df_rq2[col] = df_rq2[col].fillna('Safe').replace({'Error': 'Safe', 'API_Error': 'Safe'})
df_rq3['Prediction'] = df_rq3['Prediction'].fillna('Safe').replace({'Error': 'Safe', 'API_Error': 'Safe'})

# Sanity checks.
assert len(df_rq1) == 234, f'RQ1 CSV expected 234 rows, got {len(df_rq1)}'
assert len(df_rq2) == 234, f'RQ2 CSV expected 234 rows, got {len(df_rq2)}'
assert len(df_rq3) == 100, f'RQ3 CSV expected 100 rows, got {len(df_rq3)}'

# Cross-CSV alignment — RQ1 and RQ2 must reference the same 234 samples.
assert set(df_rq1['File_Name']) == set(df_rq2['File_Name']), 'RQ1 and RQ2 sample sets do not align.'
# RQ3 must be a subset of RQ1.
assert set(df_rq3['File_Name']).issubset(set(df_rq1['File_Name'])), 'RQ3 samples are not a subset of RQ1.'

print(f'RQ1 CSV: {len(df_rq1)} rows, {len(VARIANT_COLS_RQ1)} variants')
print(f'RQ2 CSV: {len(df_rq2)} rows, {len(VARIANT_COLS_RQ2)} variants')
print(f'RQ3 CSV: {len(df_rq3)} rows, 1 variant (gpt-3.5-turbo)')
print('All sanity checks passed.')


## 2. RQ1 main results (Table 1)

For each of the five HINTED variants and the two CLEAN re-runs, compute Accuracy, F1, Recall (sensitivity on vulnerable samples), Specificity (true negative rate on safe samples), and Precision. The pivotal column is Specificity, which spans nearly the full admissible range across variants while F1 ranges narrowly.

In [ ]:
def compute_metrics(y_true, y_pred, positive='Vulnerable', negative='Safe'):
    """Compute Accuracy, F1, Recall, Specificity, Precision.
    F1, Recall, and Precision are computed with `Vulnerable` as the positive class.
    Specificity is the true negative rate on the `Safe` class.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[positive, negative])
    tp, fn = cm[0, 0], cm[0, 1]
    fp, tn = cm[1, 0], cm[1, 1]
    return {
        'Accuracy':    accuracy_score(y_true, y_pred),
        'F1':          f1_score(y_true, y_pred, pos_label=positive, zero_division=0),
        'Recall':      recall_score(y_true, y_pred, pos_label=positive, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'Precision':   precision_score(y_true, y_pred, pos_label=positive, zero_division=0),
    }

rows = []
for variant in VARIANT_COLS_RQ1:
    m = compute_metrics(df_rq1['True_Label'], df_rq1[variant])
    rows.append({'Variant': variant + ' (HINTED)', **m})
for variant in VARIANT_COLS_RQ2:
    m = compute_metrics(df_rq2['True_Label'], df_rq2[variant])
    rows.append({'Variant': variant, **m})

table1 = pd.DataFrame(rows)
print('Table 1 — Performance of the seven prompt variants on GPT-4o-mini')
print('=' * 100)
print(table1.to_string(index=False, float_format='%.4f'))


## 3. RQ1 mechanism — Variant E (Full) vs Variant A (Baseline)

The paper's central claim for the Instruction Overload effect (Section 4.1) is that the composite Variant E regresses below the unstructured Variant A by a margin that is statistically significant under a paired McNemar test. We compute χ² and the exact p-value.

In [ ]:
def paired_mcnemar(y_true, pred_a, pred_b):
    """McNemar test on per-sample correctness of two classifiers.
    Returns (chi2 with continuity correction, exact p-value, contingency 2x2).
    """
    correct_a = (pred_a == y_true).values
    correct_b = (pred_b == y_true).values
    # 2x2 table: rows = A correct?, cols = B correct?
    n00 = int(((~correct_a) & (~correct_b)).sum())
    n01 = int(((~correct_a) &  correct_b).sum())
    n10 = int(( correct_a   & (~correct_b)).sum())
    n11 = int(( correct_a   &  correct_b).sum())
    table = np.array([[n11, n10], [n01, n00]])
    # exact=True for small counts (n01 + n10 < 25), continuity correction otherwise.
    use_exact = (n01 + n10) < 25
    result = mcnemar(table, exact=use_exact, correction=not use_exact)
    return result.statistic, result.pvalue, table, use_exact

y_true = df_rq1['True_Label']
stat, p, tbl, used_exact = paired_mcnemar(
    y_true,
    df_rq1['Variant_A_Baseline'],
    df_rq1['Variant_E_Full'],
)

print('McNemar test — Variant A (Baseline) vs Variant E (Full Framework)')
print('=' * 70)
print(f'                        E correct   E wrong')
print(f'A correct                  {tbl[0,0]:6d}    {tbl[0,1]:6d}')
print(f'A wrong                    {tbl[1,1]:6d}    {tbl[1,0]:6d}')
print()
print(f'Test type   : {"exact binomial" if used_exact else "chi-square with continuity correction"}')
print(f'Statistic   : {stat:.4f}')
print(f'p-value     : {p:.6f}')
print()
print('Cross-reference with the paper: §4.1 reports McNemar χ² = 12.97, p = 0.0003 against Baseline.')


## 4. RQ1 per-CWE breakdown (Figure 2 data)

For each variant, decompose performance by sample category: Recall on the 100 CWE-89 (SQLi) samples, Recall on the 100 CWE-78 (CmdInj) samples, and Specificity on the 34 Safe samples. This breakdown is the empirical basis for Figure 2 in the paper, and shows that aggregate F1 hides the CmdInj-Recall collapse that drives the Instruction Overload finding.

In [ ]:
def per_cwe_breakdown(df, variant_col):
    """Return a dict of {CWE-89 Recall, CWE-78 Recall, Safe Specificity} for one variant."""
    sub89 = df[df['True_CWE'] == 'CWE-89']
    sub78 = df[df['True_CWE'] == 'CWE-78']
    sub_safe = df[df['True_Label'] == 'Safe']
    return {
        'CWE-89 Recall':    (sub89[variant_col] == 'Vulnerable').mean(),
        'CWE-78 Recall':    (sub78[variant_col] == 'Vulnerable').mean(),
        'Safe Specificity': (sub_safe[variant_col] == 'Safe').mean(),
    }

rows = []
for variant in VARIANT_COLS_RQ1:
    rows.append({'Variant': variant + ' (HINTED)', **per_cwe_breakdown(df_rq1, variant)})
for variant in VARIANT_COLS_RQ2:
    rows.append({'Variant': variant, **per_cwe_breakdown(df_rq2, variant)})

fig2 = pd.DataFrame(rows)
print('Per-CWE breakdown — data underlying Figure 2')
print('=' * 90)
print(fig2.to_string(index=False, float_format='%.4f'))
print()
print('Cross-reference with the paper: Variant E HINTED CmdInj Recall drops from')
print('Variant A Baseline value of 0.96 to 0.70 — the empirical signature of Instruction Overload.')


## 5. RQ2 paired HINTED-vs-CLEAN comparison (Table 2)

For Variants C and E, compute the paired comparison of HINTED vs CLEAN forms on the same 234 samples. The McNemar test is applied to per-sample paired predictions: a sample contributes to the discordant cell if the two prompt forms produced different labels for it. The CLEAN re-runs collapse Specificity by 89% (Variant C) and 100% (Variant E) while preserving or raising Recall — the core Answer Leakage finding.

In [ ]:
# Use the correctness-based paired McNemar test (same as Section 3 above):
# samples are scored on whether each prompt produced the correct label,
# and the test compares the two classifiers' correctness profiles.

# Align RQ1 and RQ2 by File_Name to ensure paired comparison.
merged = df_rq1.merge(df_rq2, on=['File_Name', 'True_Label', 'True_CWE'], how='inner', validate='one_to_one')
assert len(merged) == 234, f'Paired merge expected 234 rows, got {len(merged)}'

rows = []
for hinted_col, clean_col, label in [
    ('Variant_C_Patterns', 'Variant_C_Patterns_CLEAN', 'Variant C (Patterns)'),
    ('Variant_E_Full',     'Variant_E_Full_CLEAN',     'Variant E (Full Framework)'),
]:
    m_hinted = compute_metrics(merged['True_Label'], merged[hinted_col])
    m_clean  = compute_metrics(merged['True_Label'], merged[clean_col])
    stat, p, tbl, used_exact = paired_mcnemar(
        merged['True_Label'], merged[hinted_col], merged[clean_col]
    )
    rows.append({
        'Variant':     label,
        'Form':        'HINTED',
        'F1':          f'{m_hinted["F1"]:.4f}',
        'Recall':      f'{m_hinted["Recall"]:.4f}',
        'Specificity': f'{m_hinted["Specificity"]:.4f}',
        'McNemar stat': f'{stat:.4f}',
        'p-value':     f'{p:.6f}',
    })
    rows.append({
        'Variant':     label,
        'Form':        'CLEAN',
        'F1':          f'{m_clean["F1"]:.4f}',
        'Recall':      f'{m_clean["Recall"]:.4f}',
        'Specificity': f'{m_clean["Specificity"]:.4f}',
        'McNemar stat': '(paired with HINTED)',
        'p-value':     '(paired with HINTED)',
    })

table2 = pd.DataFrame(rows)
print('Table 2 — Paired HINTED-vs-CLEAN comparison for Variants C and E')
print('=' * 100)
with pd.option_context('display.max_colwidth', None):
    print(table2.to_string(index=False))
print()
print('Cross-reference with the paper: Table 2 reports')
print('  Variant C: McNemar χ² = 12.96, p = 0.0003 ***')
print('  Variant E: McNemar χ² = 7.50,  p = 0.0062 **')


## 6. RQ3 cross-generation (§4.3)

GPT-3.5-turbo is evaluated on the stratified 100-sample subset using Variant C HINTED — the same prompt that achieves F1 = 0.931 on GPT-4o-mini. The headline observation is that the model classifies almost every sample as `Safe` regardless of true label, exposing a capacity floor below which prompt engineering does not operate.

In [ ]:
m_rq3 = compute_metrics(df_rq3['True_Label'], df_rq3['Prediction'])
n_safe_predictions = int((df_rq3['Prediction'] == 'Safe').sum())
n_vuln_predictions = int((df_rq3['Prediction'] == 'Vulnerable').sum())

print('GPT-3.5-turbo on Variant C (Patterns, HINTED), 100-sample stratified subset')
print('=' * 75)
print(f'Accuracy    : {m_rq3["Accuracy"]:.4f}')
print(f'Recall      : {m_rq3["Recall"]:.4f}  (sensitivity on Vulnerable samples)')
print(f'Specificity : {m_rq3["Specificity"]:.4f}  (true negative rate on Safe samples)')
print(f'F1          : {m_rq3["F1"]:.4f}')
print(f'Precision   : {m_rq3["Precision"]:.4f}')
print()
print(f'Prediction distribution: {n_safe_predictions} Safe, {n_vuln_predictions} Vulnerable (out of 100)')
print()
print('Cross-tabulation (rows = True_Label, cols = Prediction):')
print(pd.crosstab(df_rq3['True_Label'], df_rq3['Prediction']).to_string())
print()
print('Cross-reference with the paper: §4.3 reports')
print('  98 of 100 samples classified as Safe (Recall = 2/92 = 0.022, Accuracy = 0.10)')


## 7. Discordant CmdInj cases for qualitative analysis (§5.1)

The qualitative mechanism analysis in Section 5.1 of the paper inspects every CWE-78 (CmdInj) sample where Variant C and Variant E produced different predictions. We extract this set from the RQ1 results and write it to `case_study_discordant_cmdinj.csv` for downstream qualitative review.

The expected count is 30 — matching the published case_study_list.csv from the original experiments. The paper's discussion uses this set in its entirety (`all 30 discordant cases`).

In [ ]:
cmdinj = df_rq1[df_rq1['True_CWE'] == 'CWE-78'].copy()
discordant = cmdinj[cmdinj['Variant_C_Patterns'] != cmdinj['Variant_E_Full']].copy()

print(f'Total CmdInj samples            : {len(cmdinj)}')
print(f'Discordant (C != E) cases       : {len(discordant)}')
print()

if len(discordant) != 30:
    print(f'WARNING: discordant case count ({len(discordant)}) differs from published value (30). '
          f'This may indicate a difference between the rerun and the original experiment due to '
          f'temperature stochasticity at temperature={0.1}.')

print('Direction of discordance:')
direction_counts = discordant.groupby(['Variant_C_Patterns', 'Variant_E_Full']).size()
print(direction_counts.to_string())
print()
print(f'Writing case study list to {CASE_STUDY_OUT}')
discordant.to_csv(CASE_STUDY_OUT, index=False, encoding='utf-8-sig')
print('Done.')


## 8. Figure data summary

Print, in a copy-paste-friendly format, all numeric values needed to reproduce Figures 1, 2, 3, and 4 of the paper. The figures themselves are authored separately as `.drawio` source files in the `figures/` directory; this section ensures the figures and the paper text are sourced from the same canonical numbers as the rest of this notebook.

In [ ]:
print('=' * 70)
print('FIGURE 1 — F1-Score and Specificity for the seven prompt variants')
print('=' * 70)
for variant in VARIANT_COLS_RQ1:
    m = compute_metrics(df_rq1['True_Label'], df_rq1[variant])
    print(f'  {variant + " (HINTED)":40s}  F1={m["F1"]:.3f}  Spec={m["Specificity"]:.3f}')
for variant in VARIANT_COLS_RQ2:
    m = compute_metrics(df_rq2['True_Label'], df_rq2[variant])
    print(f'  {variant:40s}  F1={m["F1"]:.3f}  Spec={m["Specificity"]:.3f}')

print()
print('=' * 70)
print('FIGURE 2 — Per-CWE Recall + Safe Specificity (see Section 4 above)')
print('=' * 70)
print('Refer to the table printed in Section 4.')

print()
print('=' * 70)
print('FIGURE 3 — Paired HINTED-vs-CLEAN (see Table 2 in Section 5 above)')
print('=' * 70)
for hinted_col, clean_col, label in [
    ('Variant_C_Patterns', 'Variant_C_Patterns_CLEAN', 'Variant C (Patterns)'),
    ('Variant_E_Full',     'Variant_E_Full_CLEAN',     'Variant E (Full Framework)'),
]:
    m_h = compute_metrics(df_rq1['True_Label'], df_rq1[hinted_col])
    m_c = compute_metrics(df_rq2['True_Label'], df_rq2[clean_col])
    print(f'  {label}')
    print(f'    HINTED  F1={m_h["F1"]:.3f}  Recall={m_h["Recall"]:.3f}  Spec={m_h["Specificity"]:.3f}')
    print(f'    CLEAN   F1={m_c["F1"]:.3f}  Recall={m_c["Recall"]:.3f}  Spec={m_c["Specificity"]:.3f}')

print()
print('=' * 70)
print('FIGURE 4 — Recall-Specificity scatter (data points)')
print('=' * 70)
for variant in VARIANT_COLS_RQ1:
    m = compute_metrics(df_rq1['True_Label'], df_rq1[variant])
    print(f'  {variant + " (HINTED)":40s}  (Recall={m["Recall"]:.3f}, Spec={m["Specificity"]:.3f})')
for variant in VARIANT_COLS_RQ2:
    m = compute_metrics(df_rq2['True_Label'], df_rq2[variant])
    print(f'  {variant:40s}  (Recall={m["Recall"]:.3f}, Spec={m["Specificity"]:.3f})')
print(f'  GPT-3.5-turbo (Variant C, n=100):           (Recall={m_rq3["Recall"]:.3f}, Spec={m_rq3["Specificity"]:.3f})')
